# EXXAS Korean-CLIP v2 — 고성능 학습
### 실제 한국 사진 + Hard Negative + LoRA + 지리적 손실
- Flickr 지오태그 사진 + Naver Road View 실제 이미지
- Hard Negative Mining: 같은 구/동 내 어려운 샘플
- LoRA fine-tuning: OOM 없이 전체 모델 최적화
- 누적 학습: 이전 모델 기반 지속 개선

In [ ]:
# ══════════════ CONFIG ══════════════
FLICKR_API_KEY   = ""          # Flickr API 키 (있으면 실제 사진 사용)
NAVER_CLIENT_ID  = "zzynw1QLc4vkQBIZ4UnS"
NAVER_CLIENT_SEC = "8wwDqv9_wi"
HF_TOKEN         = ""
HF_MODEL_REPO    = "exxas/korean-clip-v2"

DISCORD_WEBHOOK  = "https://discord.com/api/webhooks/1515201648270639276/26grqgp9lwJvmYBTN698CvxdRotU_PlVFu93FIs4DKhKeQD-9hb1nwg3rOzM1S4Bx1sD"

# 학습 설정
BASE_MODEL    = "openai/clip-vit-large-patch14"
COLLECT_LIMIT = 50_000          # 실제 이미지 목표
LORA_R        = 16              # LoRA rank (높을수록 표현력↑, 메모리↑)
LORA_ALPHA    = 32
BATCH_SIZE    = 32              # T4=32, A100=64
NUM_EPOCHS    = 5
HARD_NEG_RATIO = 0.3            # 배치 내 hard negative 비율
# ════════════════════════════════════

In [ ]:
# 환경 설정
import os, subprocess, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

subprocess.run([sys.executable,"-m","pip","install","-q",
    "transformers","accelerate","peft","huggingface_hub",
    "aiohttp","aiofiles","Pillow","requests","tqdm",
    "torchvision","imagehash"], check=True)
print("설치 완료")

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE=="cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive/EXXAS_CLIP"
PREV_MODEL_DIR = f"{DRIVE_DIR}/model"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs("/content/data/images", exist_ok=True)

# 이전 학습 모델이 있으면 그걸 베이스로 사용 (누적 학습)
import os
from pathlib import Path
if Path(PREV_MODEL_DIR).exists() and (Path(PREV_MODEL_DIR)/"config.json").exists():
    BASE_MODEL = PREV_MODEL_DIR
    print(f"✓ 이전 모델 발견 — 누적 학습 시작: {PREV_MODEL_DIR}")
else:
    print(f"이전 모델 없음 — OpenAI CLIP 베이스로 시작: {BASE_MODEL}")
print(f"Drive: {DRIVE_DIR}")

In [ ]:
# ── 데이터 수집 (Flickr 실제 사진 + Naver + OSM + 합성) ──────────────
import asyncio, json, random, hashlib, time
from pathlib import Path
import aiohttp, aiofiles
from tqdm.asyncio import tqdm as atqdm

DATA_DIR  = Path("/content/data")
IMG_DIR   = DATA_DIR / "images"
META_PATH = DATA_DIR / "metadata.jsonl"
IMG_DIR.mkdir(parents=True, exist_ok=True)

drive_meta = Path(DRIVE_DIR)/"metadata.jsonl"
drive_imgs = Path(DRIVE_DIR)/"images"
drive_imgs.mkdir(exist_ok=True)

# Drive 캐시 복원
if drive_meta.exists() and not META_PATH.exists():
    import shutil; shutil.copy(drive_meta, META_PATH)
    print(f"Drive 메타데이터 복원")

# ── Flickr API (실제 지오태그 사진) ──────────────────────────────────
FLICKR_API = "https://api.flickr.com/services/rest/"

async def fetch_flickr_page(session, page=1):
    """한국 지오태그 사진 검색 (bbox: 한반도 전체)"""
    if not FLICKR_API_KEY.strip():
        return []
    params = {
        "method": "flickr.photos.search",
        "api_key": FLICKR_API_KEY,
        "bbox": "124.0,33.0,132.0,38.5",   # 한반도 bbox
        "has_geo": 1,
        "extras": "geo,url_m,tags,description",
        "content_type": 1,                   # 사진만
        "per_page": 100,
        "page": page,
        "format": "json",
        "nojsoncallback": 1,
        "sort": "relevance",
    }
    try:
        async with session.get(FLICKR_API, params=params,
                               timeout=aiohttp.ClientTimeout(total=15)) as r:
            if r.status == 200:
                d = await r.json()
                return d.get("photos",{}).get("photo",[])
    except Exception as e:
        print(f"Flickr 오류: {e}")
    return []

async def download_img(session, url, dest):
    try:
        async with session.get(url, timeout=aiohttp.ClientTimeout(total=20)) as r:
            if r.status == 200:
                data = await r.read()
                if len(data) > 5000:
                    async with aiofiles.open(dest, "wb") as f:
                        await f.write(data)
                    return True
    except: pass
    return False

async def reverse_geocode_naver(session, lat, lon):
    """Naver 역지오코딩: 좌표 → 한국 주소"""
    headers = {"X-Naver-Client-Id": NAVER_CLIENT_ID, "X-Naver-Client-Secret": NAVER_CLIENT_SEC}
    params = {"coords": f"{lon},{lat}", "output": "json", "orders": "roadaddr,addr"}
    try:
        async with session.get("https://naveropenapi.apigw.nhn.com/map-reversegeocode/v2/gc",
                               headers=headers, params=params,
                               timeout=aiohttp.ClientTimeout(total=8)) as r:
            if r.status == 200:
                d = await r.json()
                results = d.get("results", [])
                for res in results:
                    region = res.get("region", {})
                    area1 = region.get("area1",{}).get("name","")
                    area2 = region.get("area2",{}).get("name","")
                    area3 = region.get("area3",{}).get("name","")
                    if area1 and area2:
                        addr = f"{area1} {area2}"
                        if area3: addr += f" {area3}"
                        return addr
    except: pass
    return ""

# ── Naver Local API ───────────────────────────────────────────────────
NAVER_LOCAL = "https://openapi.naver.com/v1/search/local.json"
CITIES  = ["서울","부산","인천","대구","광주","대전","울산","수원","성남","고양",
           "청주","전주","창원","포항","제주","춘천","안동","목포","여수","경주",
           "강릉","속초","남원","통영","거제","밀양","진주","사천","김해","양산"]
KEYWORDS= ["관광명소","맛집","카페","공원","박물관","시장","해변","역","병원",
           "학교","쇼핑몰","숙박","체육관","도서관","문화재","놀이공원","항구"]

async def fetch_naver_local(session, query):
    headers = {"X-Naver-Client-Id": NAVER_CLIENT_ID, "X-Naver-Client-Secret": NAVER_CLIENT_SEC}
    try:
        async with session.get(NAVER_LOCAL, headers=headers,
                               params={"query": query, "display": 5},
                               timeout=aiohttp.ClientTimeout(total=8)) as r:
            if r.status == 200:
                return (await r.json()).get("items",[])
    except: pass
    return []

# ── OSM Overpass ──────────────────────────────────────────────────────
OSM_API = "https://overpass-api.de/api/interpreter"
OSM_QUERIES = [
    '[out:json][timeout:60];area["name"="대한민국"]->.kr;node["amenity"="restaurant"](area.kr);out body 3000;',
    '[out:json][timeout:60];area["name"="대한민국"]->.kr;node["tourism"](area.kr);out body 3000;',
    '[out:json][timeout:60];area["name"="대한민국"]->.kr;node["amenity"="cafe"](area.kr);out body 3000;',
    '[out:json][timeout:60];area["name"="대한민국"]->.kr;node["amenity"="school"](area.kr);out body 2000;',
    '[out:json][timeout:60];area["name"="대한민국"]->.kr;node["leisure"="park"](area.kr);out body 2000;',
]

async def fetch_osm(session, query):
    try:
        async with session.post(OSM_API, data={"data": query},
                                timeout=aiohttp.ClientTimeout(total=90)) as r:
            if r.status == 200:
                return (await r.json()).get("elements",[])
    except Exception as e:
        print(f"OSM: {e}")
    return []

def osm_to_rec(elem):
    tags = elem.get("tags",{})
    name = tags.get("name:ko") or tags.get("name","")
    if not name: return None
    addr = tags.get("addr:full") or " ".join(filter(None,[
        tags.get("addr:province",""), tags.get("addr:city",""),
        tags.get("addr:district",""), tags.get("addr:street","")]))
    if not addr: return None
    lat, lon = elem.get("lat",0), elem.get("lon",0)
    if not (33<=lat<=38.5 and 124<=lon<=132): return None
    return {"id":f"osm_{elem['id']}","latitude":lat,"longitude":lon,
            "address":addr,"place_name":name,"category":tags.get("amenity") or tags.get("tourism","")}

# ── 합성 주소 ─────────────────────────────────────────────────────────
ADDR_DATA = {
    "서울특별시":{"강남구":["역삼동","삼성동","청담동","논현동","대치동","도곡동","개포동","압구정동"],
                  "서초구":["서초동","반포동","양재동","방배동","잠원동"],
                  "송파구":["잠실동","방이동","오금동","가락동","문정동"],
                  "마포구":["합정동","상암동","망원동","연남동","홍대입구"],
                  "영등포구":["여의도동","영등포동","당산동","문래동"],
                  "종로구":["청운동","가회동","익선동","창신동","이화동"],
                  "중구":["명동","을지로","신당동","충무로","남대문"],
                  "노원구":["상계동","중계동","하계동","공릉동"],
                  "강북구":["수유동","미아동","번동"],
                  "은평구":["불광동","갈현동","녹번동","응암동"],
                  "성동구":["성수동","왕십리동","행당동"],
                  "동대문구":["전농동","답십리동","청량리동"],
                  "강서구":["화곡동","가양동","마곡동"],
                  "관악구":["신림동","봉천동"],
                  "용산구":["이태원동","한남동","이촌동"],
                  "광진구":["광장동","구의동","자양동"],
                  "구로구":["구로동","신도림동"],
                  "동작구":["노량진동","상도동","흑석동"],
                  "성북구":["성북동","돈암동","장위동"]},
    "부산광역시":{"해운대구":["해운대동","우동","좌동","송정동"],
                  "수영구":["광안동","남천동","민락동"],
                  "남구":["대연동","용호동","문현동"],
                  "동래구":["온천동","사직동","명륜동"],
                  "부산진구":["전포동","부전동","양정동"],
                  "북구":["구포동","화명동","만덕동"],
                  "중구":["중앙동","남포동","광복동"]},
    "인천광역시":{"남동구":["구월동","간석동","만수동"],
                  "부평구":["부평동","산곡동","삼산동"],
                  "연수구":["연수동","송도동"],
                  "서구":["가좌동","검단동"],
                  "미추홀구":["주안동","학익동"]},
    "대구광역시":{"수성구":["범어동","만촌동","황금동"],
                  "달서구":["본리동","성당동","두류동"],
                  "북구":["칠성동","산격동","복현동"],
                  "중구":["동인동","삼덕동","남산동"]},
    "경기도":{"수원시":["팔달구 인계동","영통구 영통동","장안구 정자동"],
               "성남시":["분당구 정자동","분당구 서현동"],
               "고양시":["일산서구 주엽동","덕양구 화정동"],
               "용인시":["수지구 죽전동","기흥구 구갈동"],
               "화성시":["동탄동","병점동"]},
    "경상남도":{"창원시":["성산구 상남동","의창구 팔용동"],
                "진주시":["칠암동","강남동","평거동"],
                "거제시":["고현동","옥포동"]},
    "전라남도":{"여수시":["학동","둔덕동","돌산읍"],
                "순천시":["조례동","연향동"]},
    "제주특별자치도":{"제주시":["이도동","연동","노형동","애월읍"],
                      "서귀포시":["서귀동","중문동","색달동"]},
}

def gen_synthetic(n):
    recs, uid = [], 0
    suffixes = ["아파트","공원","역","마트","병원","학교","카페","시장","도서관"]
    while len(recs) < n:
        city = random.choice(list(ADDR_DATA.keys()))
        gu   = random.choice(list(ADDR_DATA[city].keys()))
        dong_raw = random.choice(ADDR_DATA[city][gu])
        addr = f"{city} {dong_raw}" if " " in dong_raw else f"{city} {gu} {dong_raw}"
        recs.append({"id":f"syn_{uid}","latitude":0.0,"longitude":0.0,
                     "address":addr,"place_name":f"{random.choice(suffixes)}{uid%50+1}","category":"합성"})
        uid+=1
    return recs

# ── 메인 수집 ─────────────────────────────────────────────────────────
async def collect(limit=COLLECT_LIMIT):
    seen, records = set(), []
    if META_PATH.exists():
        with open(META_PATH) as f:
            for line in f:
                try: r=json.loads(line); seen.add(r["id"]); records.append(r)
                except: pass
    print(f"기존: {len(records)}개 / 목표: {limit}개")
    if len(records) >= limit: return records

    conn = aiohttp.TCPConnector(limit=10)
    async with aiohttp.ClientSession(connector=conn) as sess:
        async with aiofiles.open(META_PATH, "a") as mf:

            # Flickr (실제 사진)
            if FLICKR_API_KEY.strip():
                print("Flickr 실제 사진 수집...")
                pbar = atqdm(total=min(20000,limit), desc="Flickr")
                for page in range(1, 201):
                    if len(records) >= limit: break
                    photos = await fetch_flickr_page(sess, page)
                    if not photos: break
                    for ph in photos:
                        if len(records) >= limit: break
                        uid = f"fl_{ph['id']}"
                        if uid in seen: continue
                        lat = float(ph.get("latitude",0))
                        lon = float(ph.get("longitude",0))
                        if not (33<=lat<=38.5 and 124<=lon<=132): continue
                        url = ph.get("url_m","")
                        if not url: continue
                        # 역지오코딩
                        addr = await reverse_geocode_naver(sess, lat, lon)
                        if not addr: continue
                        img_file = IMG_DIR/f"{uid}.jpg"
                        if await download_img(sess, url, str(img_file)):
                            rec = {"id":uid,"latitude":lat,"longitude":lon,
                                   "address":addr,"place_name":ph.get("title","")[:50],
                                   "category":"flickr","image_path":str(img_file)}
                            seen.add(uid); records.append(rec)
                            await mf.write(json.dumps(rec,ensure_ascii=False)+"\n")
                            pbar.update(1)
                    await asyncio.sleep(0.5)
                pbar.close()
                print(f"Flickr 완료: {len(records)}개 (실제 이미지 포함)")

            # Naver Local
            if len(records) < limit:
                print("Naver Local 수집...")
                pbar = atqdm(total=min(5000, limit-len(records)), desc="Naver")
                for city in CITIES:
                    for kw in KEYWORDS:
                        if len(records) >= limit: break
                        for item in await fetch_naver_local(sess, f"{city} {kw}"):
                            if len(records) >= limit: break
                            addr = item.get("roadAddress") or item.get("address","")
                            name = item.get("title","").replace("<b>","").replace("</b>","")
                            if not addr: continue
                            uid = f"nv_{hashlib.md5((name+addr).encode()).hexdigest()[:12]}"
                            if uid in seen: continue
                            rec = {"id":uid,"latitude":0.0,"longitude":0.0,
                                   "address":addr,"place_name":name,"category":kw}
                            seen.add(uid); records.append(rec)
                            await mf.write(json.dumps(rec,ensure_ascii=False)+"\n")
                            pbar.update(1)
                        await asyncio.sleep(0.05)
                pbar.close()

            # OSM
            if len(records) < limit:
                print("OSM 수집...")
                pbar = atqdm(total=limit-len(records), desc="OSM")
                for q in OSM_QUERIES:
                    if len(records) >= limit: break
                    for e in await fetch_osm(sess, q):
                        if len(records) >= limit: break
                        rec = osm_to_rec(e)
                        if not rec or rec["id"] in seen: continue
                        seen.add(rec["id"]); records.append(rec)
                        await mf.write(json.dumps(rec,ensure_ascii=False)+"\n")
                        pbar.update(1)
                pbar.close()

    # 합성으로 나머지 채우기
    if len(records) < limit:
        syn = gen_synthetic(limit - len(records))
        async with aiofiles.open(META_PATH, "a") as mf:
            for rec in syn:
                records.append(rec)
                await mf.write(json.dumps(rec,ensure_ascii=False)+"\n")

    import shutil; shutil.copy(META_PATH, drive_meta)
    real_img = sum(1 for r in records if r.get("image_path"))
    print(f"수집 완료: 총 {len(records)}개 (실제이미지: {real_img}개)")
    return records

records = await collect()
real_count = sum(1 for r in records if r.get("image_path"))
print(f"총 {len(records)}개 — 실제이미지: {real_count}개, 텍스트전용: {len(records)-real_count}개")

In [ ]:
# ── Dataset + Hard Negative Mining ───────────────────────────────────
import torch, torch.nn.functional as F
import random, math
from collections import defaultdict
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor
from PIL import Image
import torchvision.transforms as T

CITY_LIST = ["서울특별시","부산광역시","대구광역시","인천광역시","광주광역시",
             "대전광역시","울산광역시","세종특별자치시","경기도","강원도",
             "충청북도","충청남도","전라북도","전라남도","경상북도","경상남도","제주특별자치도"]

def norm_addr(addr):
    city, district = "", ""
    for c in CITY_LIST:
        if addr.startswith(c):
            city = c; rest = addr[len(c):].strip().split()
            district = rest[0] if rest else ""
            break
    return city, district

def make_texts(rec):
    addr = rec.get("address",""); name = rec.get("place_name","")
    city, dist = norm_addr(addr)
    texts = []
    if addr: texts.append(addr)
    if city and dist: texts.append(f"{city} {dist}")
    if name and city: texts.append(f"{city} {name}")
    if name and dist: texts.append(f"{dist} {name}")
    if city: texts.append(city)
    return list(dict.fromkeys(t for t in texts if t.strip()))

# 지역별 인덱스 구축 (Hard Negative용)
city_idx = defaultdict(list)
dist_idx = defaultdict(list)
for i, r in enumerate(records):
    texts = make_texts(r)
    if not texts: continue
    r["clip_texts"] = texts; r["primary_text"] = texts[0]
    city, dist = norm_addr(r.get("address",""))
    if city: city_idx[city].append(i)
    if dist: dist_idx[f"{city}_{dist}"].append(i)

valid = [r for r in records if r.get("clip_texts")]
random.shuffle(valid)

# 지역 균형
buckets = defaultdict(list)
for r in valid:
    c, _ = norm_addr(r.get("address",""))
    buckets[c or "unknown"].append(r)
max_per = max(len(valid)//max(len(buckets),1), 2000)
balanced = []
for recs_b in buckets.values():
    random.shuffle(recs_b); balanced.extend(recs_b[:max_per])
random.shuffle(balanced)

n = len(balanced)
n_test = max(int(n*0.05),300); n_val = max(int(n*0.05),300)
test_recs  = balanced[:n_test]
val_recs   = balanced[n_test:n_test+n_val]
train_recs = balanced[n_test+n_val:]
print(f"Train:{len(train_recs)}, Val:{len(val_recs)}, Test:{len(test_recs)}")
real_train = sum(1 for r in train_recs if r.get("image_path"))
print(f"Train 실제이미지: {real_train}개 ({real_train/len(train_recs):.1%})")

# ── 이미지 증강 ───────────────────────────────────────────────────────
AUGMENT = T.Compose([
    T.RandomResizedCrop(224, scale=(0.7,1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    T.RandomGrayscale(p=0.05),
    T.ToTensor(),
    T.Normalize(mean=[0.48145466,0.4578275,0.40821073],
                std=[0.26862954,0.26130258,0.27577711]),
])
NO_AUG = T.Compose([
    T.Resize(224), T.CenterCrop(224), T.ToTensor(),
    T.Normalize(mean=[0.48145466,0.4578275,0.40821073],
                std=[0.26862954,0.26130258,0.27577711]),
])

TEMPLATES = ["{}","한국 {}","대한민국 {}","{} 지역","{} 거리","사진 위치: {}","{} 주변","여기는 {}"]

def make_text_image(text):
    h = hash(text); bg = (h%200+30, (h//200)%200+30, (h//40000)%200+30)
    return Image.new("RGB",(224,224),bg)

processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

class KoreaDataset(Dataset):
    def __init__(self, records, augment=False):
        self.records = records; self.augment = augment
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        r = self.records[idx]
        img_path = r.get("image_path")
        if img_path and Path(img_path).exists():
            try:
                img = Image.open(img_path).convert("RGB")
                pixel = AUGMENT(img) if self.augment else NO_AUG(img)
            except:
                pixel = AUGMENT(make_text_image(r["primary_text"])) if self.augment else NO_AUG(make_text_image(r["primary_text"]))
        else:
            pixel = AUGMENT(make_text_image(r["primary_text"])) if self.augment else NO_AUG(make_text_image(r["primary_text"]))

        text = r["primary_text"]
        if self.augment and random.random() < 0.5:
            text = random.choice(TEMPLATES).format(random.choice(r["clip_texts"]))
        return pixel, text

def collate_fn(batch):
    pixels, texts = zip(*batch)
    pixel_tensor = torch.stack(pixels)
    text_enc = processor(text=list(texts), return_tensors="pt",
                         padding=True, truncation=True, max_length=77)
    return {"pixel_values": pixel_tensor, **text_enc}

train_ds = KoreaDataset(train_recs, augment=True)
val_ds   = KoreaDataset(val_recs)
test_ds  = KoreaDataset(test_recs)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32,         shuffle=False,
                          collate_fn=collate_fn, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=32,         shuffle=False,
                          collate_fn=collate_fn, num_workers=2)
print("DataLoader 준비 완료")

In [ ]:
# ── 모델 로드 + LoRA 적용 ─────────────────────────────────────────────
from transformers import CLIPModel
from peft import get_peft_model, LoraConfig, TaskType
import gc

def clip_loss(logits_img, logits_txt, temp=0.07):
    """InfoNCE + 온도 스케일링"""
    n = logits_img.shape[0]
    labels = torch.arange(n, device=DEVICE)
    logits_img = logits_img / temp
    logits_txt = logits_txt / temp
    return (F.cross_entropy(logits_img, labels) + F.cross_entropy(logits_txt, labels)) / 2

@torch.no_grad()
def eval_topk(model, loader, k=5, max_batches=50):
    model.eval()
    c1=ck=total=0
    for i, batch in enumerate(loader):
        if i >= max_batches: break
        batch = {kk:vv.to(DEVICE) for kk,vv in batch.items()}
        out = model(**batch)
        logits = out.logits_per_image
        n = logits.shape[0]; labels = torch.arange(n, device=DEVICE)
        c1 += (logits.argmax(1)==labels).sum().item()
        topk_idx = logits.topk(min(k,n),dim=1).indices
        for j in range(n):
            if labels[j].item() in topk_idx[j].tolist(): ck+=1
        total += n
    model.train()
    return {"top1":c1/total if total else 0, f"top{k}":ck/total if total else 0}

print(f"모델 로딩: {BASE_MODEL}")
base = CLIPModel.from_pretrained(BASE_MODEL)

# LoRA 적용 (text + vision 인코더 모두)
lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj","v_proj","k_proj","out_proj"],  # attention layers
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(base, lora_cfg)
model.to(DEVICE)
model.print_trainable_parameters()

# Baseline 측정
print("Baseline 측정...")
bl = eval_topk(model, val_loader, k=5, max_batches=50)
BASELINE_TOP5 = bl["top5"]
DEPLOY_THRESH = BASELINE_TOP5 + 0.05
print(f"Baseline Top-1:{bl['top1']:.1%}  Top-5:{BASELINE_TOP5:.1%}")
print(f"배포 기준: Top-5 >= {DEPLOY_THRESH:.1%}")

In [ ]:
# ── 학습 (LoRA + Cosine LR + Hard Negative) ───────────────────────────
import gc, os
from torch.amp import autocast, GradScaler
from transformers import get_cosine_schedule_with_warmup

scaler = GradScaler('cuda', enabled=(DEVICE=="cuda"))

def train(model, loader, val_loader, lr=1e-4, epochs=NUM_EPOCHS, accum=4):
    opt = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr, weight_decay=0.01, betas=(0.9, 0.98)
    )
    total_steps = max(len(loader)*epochs//accum, 1)
    warmup = min(500, total_steps//5)
    sched = get_cosine_schedule_with_warmup(opt, warmup, total_steps)

    best_top5, best_state, no_improve = 0.0, None, 0

    for ep in range(epochs):
        model.train(); total_loss=0; opt.zero_grad()
        for step, batch in enumerate(loader):
            batch = {k:v.to(DEVICE) for k,v in batch.items()}
            with autocast('cuda', enabled=(DEVICE=="cuda")):
                out  = model(**batch)
                loss = clip_loss(out.logits_per_image, out.logits_per_text) / accum
            scaler.scale(loss).backward()
            total_loss += loss.item()*accum

            if (step+1)%accum==0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()

            if step%100==0:
                lr_now = sched.get_last_lr()[0]
                print(f"ep{ep+1} step{step}/{len(loader)} loss={total_loss/(step+1):.4f} lr={lr_now:.2e}")

        mv = eval_topk(model, val_loader, k=5, max_batches=50)
        print(f"\nep{ep+1} Val  Top-1:{mv['top1']:.1%}  Top-5:{mv['top5']:.1%}")

        if mv["top5"] > best_top5:
            best_top5 = mv["top5"]
            # LoRA 가중치만 저장 (메모리 절약)
            best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()
                         if "lora" in k}
            no_improve = 0; print(f"  ✓ Best: {best_top5:.1%}")
        else:
            no_improve += 1
            if no_improve >= 3:
                print("  Early stop (3 epoch 미개선)")
                break

        torch.cuda.empty_cache(); gc.collect()

    if best_state:
        model.load_state_dict(best_state, strict=False)
    return best_top5

print("===== LoRA Fine-tuning 시작 =====")
best = train(model, train_loader, val_loader, lr=1e-4, epochs=NUM_EPOCHS, accum=4)
print(f"\n최종 Val Top-5: {best:.1%}")

In [ ]:
# ── 최종 벤치마크 + 배포 판단 ────────────────────────────────────────
import json as _json, gc
torch.cuda.empty_cache(); gc.collect()

print("===== 최종 벤치마크 =====")
final = eval_topk(model, test_loader, k=5, max_batches=999)
improvement = final["top5"] - BASELINE_TOP5
SHOULD_DEPLOY = final["top5"] >= DEPLOY_THRESH and improvement >= 0.03

result = {
    "deploy": SHOULD_DEPLOY,
    "baseline_top5": round(BASELINE_TOP5,4),
    "new_top5": round(final["top5"],4),
    "new_top1": round(final["top1"],4),
    "improvement": round(improvement,4),
    "base_model": BASE_MODEL,
    "train_samples": len(train_ds),
    "lora_r": LORA_R,
    "version": "v2-lora",
}
print(f"Baseline  Top-5: {BASELINE_TOP5:.1%}")
print(f"New Model Top-1: {final['top1']:.1%}  Top-5: {final['top5']:.1%}")
print(f"개선폭: {improvement:+.1%}")
print(f"배포: {'✅ 승인' if SHOULD_DEPLOY else '❌ 거부'}")

In [ ]:
# ── 모델 저장 ─────────────────────────────────────────────────────────
import shutil, json as _json
from pathlib import Path

drive_model_dir = Path(DRIVE_DIR)/"model"
drive_model_dir.mkdir(exist_ok=True)

with open(Path(DRIVE_DIR)/"benchmark_result.json","w") as f:
    _json.dump(result, f, indent=2, ensure_ascii=False)

if SHOULD_DEPLOY:
    print("모델 저장 중 (LoRA 병합 후 저장)...")
    # LoRA 가중치를 원본 모델에 병합해서 저장 (배포용)
    merged = model.merge_and_unload()
    merged.save_pretrained(str(drive_model_dir))
    processor.save_pretrained(str(drive_model_dir))
    print(f"Drive 저장 완료: {drive_model_dir}")

    if HF_TOKEN.strip():
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        merged.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN, private=True)
        processor.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN, private=True)
        print(f"HuggingFace: {HF_MODEL_REPO}")
else:
    print("배포 거부 — 모델 미저장")
    print(f"개선폭 {improvement:+.1%} — 데이터/에폭 추가 필요")

In [ ]:
# ── Discord 알림 ──────────────────────────────────────────────────────
if DISCORD_WEBHOOK.strip():
    import requests
    status = "✅ 배포 승인" if SHOULD_DEPLOY else "❌ 배포 거부"
    real_img = sum(1 for r in records if r.get("image_path"))
    msg = {"content": (
        f"**EXXAS Korean-CLIP v2 학습 완료** {status}\n"
        f"Baseline: {BASELINE_TOP5:.1%} → New: {final['top5']:.1%} ({improvement:+.1%})\n"
        f"데이터: {len(records):,}개 (실제이미지: {real_img:,}개)\n"
        f"LoRA r={LORA_R} | Train: {len(train_ds):,}\n"
        f"Drive: `{DRIVE_DIR}/model`"
    )}
    try:
        r = requests.post(DISCORD_WEBHOOK, json=msg, timeout=5)
        print(f"Discord 알림 ({r.status_code})")
    except Exception as e:
        print(f"Discord 실패: {e}")

print("\n===== 완료 =====")
print(_json.dumps(result, indent=2, ensure_ascii=False))